In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import zipfile
import os
zip_path = "/content/tinyllama-instruction.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"

In [ ]:
import pip
import pkg_resources
try:
    # Check for torchao version and upgrade if necessary
    torchao_version = pkg_resources.get_distribution("torchao").version
    if pkg_resources.parse_version(torchao_version) < pkg_resources.parse_version("0.16.0"):
        print(f"Upgrading torchao from {torchao_version} to a version >=0.16.0...")
        !pip install torchao --upgrade --quiet
except pkg_resources.DistributionNotFound:
    print("torchao not found, installing...")
    !pip install torchao --quiet

instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U bitsandbytes

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
instruction_checkpoint = "/content/checkpoint-3"

In [ ]:
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [ ]:
# model = model.merge_and_unload()

In [ ]:
pref_model_lora = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=[],
    logging_dir=None,
    loss_type="sigmoid",
    remove_unused_columns=False
)

In [ ]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 6.2698, 'train_samples_per_second': 0.797, 'train_steps_per_second': 0.159, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996, 'entropy': 2.0842321634292604, 'num_tokens': 566.0, 'logits/chosen': -3.5440226682366416, 'logits/rejected': -3.6391777078549863, 'mean_token_accuracy': 0.5961200177669526, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -93.67644958496093, 'logps/rejected': -62.53306274414062, 'epoch': 1.0})

In [ ]:
question = "how does Metformin works in the human and does it have benefits for diabetes patient?"

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
zip_path = "/content/tinyllama-non-instruction.zip"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()


In [ ]:
model_path = "/content/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization


In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

how does Metformin works in the human and does it have benefits for diabetes patient?
How to take metformin 850mg?
I have taken metformin 500 mg twice a day since last week. What is the maximum dose I should be taking at any given time? How much food should I eat when I am taking Metformin?
I had my first blood test yesterday and everything was fine, but today, they took my blood again. It seems like it's been 2 months that I didn't


In [ ]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    device_map="auto"
)

preference_aligned_model = PeftModel.from_pretrained(base_model_for_inference, model_path)

In [ ]:
preference_aligned_model.to("cuda")

In [ ]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))